In [ ]:
# --- STEP 0: Install deps (if not already) ---
import pandas as pd
import numpy as np
from scipy.signal import resample
from scipy.stats import skew, kurtosis
from scipy.signal import find_peaks
from scipy.fft import fft


In [ ]:
# --- STEP 1: Load dataset ---

df = pd.read_csv("/content/drive/MyDrive/Defence/BP_Measuring_finalDataV2.csv")
display(df.head(10))

,_id,serialNumber,ir[0],ir[1],ir[2],ir[3],ir[4],ir[5],ir[6],ir[7],...,red[169],spo2,heartRate,userAge,userGender,userHeight,userWeight,userBmi,systolic_BP,diastolic_BP
0,689666a4054bb9ca6f371a48,1,53465,53297,53166,53039,53234.0,53247.0,53156.0,53171.0,...,NaN,100,107,44,Male,180.0,98,30.246914,122,83
1,689666a4054bb9ca6f371a49,2,54755,54760,54767,54775,54782.0,54785.0,54804.0,54817.0,...,30759.0,100,125,30,Female,154.0,43,18.131219,102,68
2,689666a4054bb9ca6f371a4a,3,61920,61724,61878,61801,61611.0,61651.0,61801.0,61822.0,...,NaN,100,93,61,Female,144.0,62,29.899691,141,93
3,689666a4054bb9ca6f371a4b,4,59870,59829,59868,59922,59982.0,59954.0,59904.0,59956.0,...,NaN,98,71,45,Male,172.0,103,34.816117,139,96
4,689666a4054bb9ca6f371a4c,5,61487,61599,61710,61695,61531.0,61550.0,61506.0,61582.0,...,45313.0,98,166,23,Male,172.0,88,29.745809,112,75
5,689666a4054bb9ca6f371a4d,6,61878,62116,62117,62150,62085.0,62098.0,62093.0,62133.0,...,NaN,100,107,21,Male,165.0,68,24.977043,104,61
6,689666a4054bb9ca6f371a4e,7,55798,55766,55710,55678,55677.0,55689.0,55689.0,55698.0,...,NaN,100,136,22,Male,167.0,86,30.836531,108,80
7,689666a4054bb9ca6f371a4f,8,65781,65850,65149,65549,65698.0,65876.0,65877.0,65411.0,...,NaN,100,150,23,Male,177.0,107,34.153660,111,72
8,689666a4054bb9ca6f371a50,9,62554,2783,1136,1148,24925.0,58433.0,59563.0,60109.0,...,NaN,99,68,21,Male,182.0,100,30.189591,120,79
9,689666a4054bb9ca6f371a51,10,1024,1023,1025,1026,NaN,NaN,NaN,NaN,...,NaN,98,72,30,Male,175.0,80,26.122449,120,80


In [ ]:
# Drop the row with index 9
df = df.drop(9)

print("Row with index 9 dropped.")
print(f"New DataFrame shape: {df.shape}")

Row with index 9 dropped.
New DataFrame shape: (328, 351)


In [ ]:
display(df.head(10))

,_id,serialNumber,ir[0],ir[1],ir[2],ir[3],ir[4],ir[5],ir[6],ir[7],...,red[169],spo2,heartRate,userAge,userGender,userHeight,userWeight,userBmi,systolic_BP,diastolic_BP
0,689666a4054bb9ca6f371a48,1,53465,53297,53166,53039,53234.0,53247.0,53156.0,53171.0,...,NaN,100,107,44,Male,180.0,98,30.246914,122,83
1,689666a4054bb9ca6f371a49,2,54755,54760,54767,54775,54782.0,54785.0,54804.0,54817.0,...,30759.0,100,125,30,Female,154.0,43,18.131219,102,68
2,689666a4054bb9ca6f371a4a,3,61920,61724,61878,61801,61611.0,61651.0,61801.0,61822.0,...,NaN,100,93,61,Female,144.0,62,29.899691,141,93
3,689666a4054bb9ca6f371a4b,4,59870,59829,59868,59922,59982.0,59954.0,59904.0,59956.0,...,NaN,98,71,45,Male,172.0,103,34.816117,139,96
4,689666a4054bb9ca6f371a4c,5,61487,61599,61710,61695,61531.0,61550.0,61506.0,61582.0,...,45313.0,98,166,23,Male,172.0,88,29.745809,112,75
5,689666a4054bb9ca6f371a4d,6,61878,62116,62117,62150,62085.0,62098.0,62093.0,62133.0,...,NaN,100,107,21,Male,165.0,68,24.977043,104,61
6,689666a4054bb9ca6f371a4e,7,55798,55766,55710,55678,55677.0,55689.0,55689.0,55698.0,...,NaN,100,136,22,Male,167.0,86,30.836531,108,80
7,689666a4054bb9ca6f371a4f,8,65781,65850,65149,65549,65698.0,65876.0,65877.0,65411.0,...,NaN,100,150,23,Male,177.0,107,34.153660,111,72
8,689666a4054bb9ca6f371a50,9,62554,2783,1136,1148,24925.0,58433.0,59563.0,60109.0,...,NaN,99,68,21,Male,182.0,100,30.189591,120,79
10,689666a4054bb9ca6f371a52,11,59696,59669,59694,59683,59732.0,59569.0,59619.0,59619.0,...,NaN,100,115,21,Male,180.0,78,24.074074,116,68


In [ ]:
# Identify entries with -999 in heartRate or spo2
error_entries = df[(df['heartRate'] == -999) | (df['spo2'] == -999)]

# Display the identified entries
display(error_entries)

# Print the number of such entries
print(f"\nNumber of entries with -999 in heartRate or spo2: {len(error_entries)}")

,_id,serialNumber,ir[0],ir[1],ir[2],ir[3],ir[4],ir[5],ir[6],ir[7],...,red[169],spo2,heartRate,userAge,userGender,userHeight,userWeight,userBmi,systolic_BP,diastolic_BP
25,689666a4054bb9ca6f371a61,26,53864,53871,53839,53700,53666.0,53673.0,53704.0,53683.0,...,NaN,-999,187,22,Male,167.00,70,25.099502,116,74
53,68971b8c6af5bb099f84c693,54,52329,52255,52322,52279,52157.0,52139.0,52285.0,52321.0,...,NaN,-999,88,21,Female,160.02,55,21.480000,112,74
59,6897385157f90f6e80917148,60,56938,56954,56909,56921,56961.0,56968.0,56911.0,56969.0,...,NaN,-999,115,25,Male,170.18,76,26.240000,120,80
141,6898ce537e9100dfd3236ed0,142,52901,52895,52882,52882,52859.0,52845.0,52849.0,52845.0,...,NaN,-999,-999,22,Male,180.34,69,21.220000,108,75
149,6898d1b77e9100dfd3236ee0,150,52939,52974,52996,52782,52887.0,52967.0,52987.0,52952.0,...,NaN,-999,-999,25,Male,160.02,72,28.120000,112,71
152,6898d3087e9100dfd3236ee7,153,53179,53191,53197,53228,53194.0,53181.0,53207.0,53173.0,...,NaN,-999,187,23,Male,165.10,68,24.950000,132,83
181,6898e2e57e9100dfd3236f22,182,54833,54812,54799,54771,54762.0,54747.0,54765.0,54778.0,...,NaN,-999,-999,24,Male,165.10,56,20.540000,119,73
184,6898e4987e9100dfd3236f29,185,51377,51398,51376,51379,51340.0,51333.0,51317.0,51286.0,...,NaN,-999,-999,23,Male,177.80,58,18.350000,109,71
195,6898e9ff8b93e8da8a1e52ff,196,59047,59100,59002,59062,59058.0,59113.0,59148.0,59159.0,...,NaN,-999,-999,21,Male,182.88,85,25.410000,114,69
211,6899a52ce986f6946ce7d422,212,55206,55226,55206,55219,55208.0,55258.0,55314.0,55299.0,...,NaN,-999,65,25,Male,175.26,70,22.790000,114,77



Number of entries with -999 in heartRate or spo2: 17


In [ ]:
# Identify entries with -999 in heartRate
error_entries_HR = df[(df['heartRate'] == -999)]

# Display the identified entries
display(error_entries_HR)

# Print the number of such entries
print(f"\nNumber of entries with -999 in heartRate {len(error_entries_HR)}")

,_id,serialNumber,ir[0],ir[1],ir[2],ir[3],ir[4],ir[5],ir[6],ir[7],...,red[169],spo2,heartRate,userAge,userGender,userHeight,userWeight,userBmi,systolic_BP,diastolic_BP
141,6898ce537e9100dfd3236ed0,142,52901,52895,52882,52882,52859.0,52845.0,52849.0,52845.0,...,NaN,-999,-999,22,Male,180.34,69,21.22,108,75
149,6898d1b77e9100dfd3236ee0,150,52939,52974,52996,52782,52887.0,52967.0,52987.0,52952.0,...,NaN,-999,-999,25,Male,160.02,72,28.12,112,71
181,6898e2e57e9100dfd3236f22,182,54833,54812,54799,54771,54762.0,54747.0,54765.0,54778.0,...,NaN,-999,-999,24,Male,165.10,56,20.54,119,73
184,6898e4987e9100dfd3236f29,185,51377,51398,51376,51379,51340.0,51333.0,51317.0,51286.0,...,NaN,-999,-999,23,Male,177.80,58,18.35,109,71
195,6898e9ff8b93e8da8a1e52ff,196,59047,59100,59002,59062,59058.0,59113.0,59148.0,59159.0,...,NaN,-999,-999,21,Male,182.88,85,25.41,114,69
236,6899b3b9e986f6946ce7d45a,237,54231,54219,54217,54213,54192.0,54204.0,54195.0,54202.0,...,NaN,-999,-999,21,Female,165.10,70,25.68,115,85
241,6899b6bce986f6946ce7d466,242,59814,59802,59822,59806,59809.0,59843.0,59824.0,59841.0,...,NaN,-999,-999,21,Female,165.10,70,25.68,115,85
263,689a2934a5d2fda0e473600d,264,56060,56038,56024,55973,55935.0,55935.0,55925.0,55945.0,...,NaN,-999,-999,23,Male,175.26,69,22.46,117,81
298,68a1f54485ff9a6c818171d0,299,54205,54229,54048,54205,54010.0,53190.0,53275.0,53312.0,...,NaN,-999,-999,23,Male,165.10,60,22.01,118,87
307,68a200f3d5548939d4f812be,308,46915,48095,47731,47030,47246.0,47704.0,48362.0,48664.0,...,NaN,-999,-999,24,Male,165.10,72,26.41,133,91



Number of entries with -999 in heartRate 10


In [ ]:
# Identify entries with -999 in heartRate or spo2
error_entries_SPO2 = df[(df['spo2'] == -999)]

# Display the identified entries
display(error_entries_SPO2)

# Print the number of such entries
print(f"\nNumber of entries with -999 in or spo2: {len(error_entries_SPO2)}")

,_id,serialNumber,ir[0],ir[1],ir[2],ir[3],ir[4],ir[5],ir[6],ir[7],...,red[169],spo2,heartRate,userAge,userGender,userHeight,userWeight,userBmi,systolic_BP,diastolic_BP
25,689666a4054bb9ca6f371a61,26,53864,53871,53839,53700,53666.0,53673.0,53704.0,53683.0,...,NaN,-999,187,22,Male,167.00,70,25.099502,116,74
53,68971b8c6af5bb099f84c693,54,52329,52255,52322,52279,52157.0,52139.0,52285.0,52321.0,...,NaN,-999,88,21,Female,160.02,55,21.480000,112,74
59,6897385157f90f6e80917148,60,56938,56954,56909,56921,56961.0,56968.0,56911.0,56969.0,...,NaN,-999,115,25,Male,170.18,76,26.240000,120,80
141,6898ce537e9100dfd3236ed0,142,52901,52895,52882,52882,52859.0,52845.0,52849.0,52845.0,...,NaN,-999,-999,22,Male,180.34,69,21.220000,108,75
149,6898d1b77e9100dfd3236ee0,150,52939,52974,52996,52782,52887.0,52967.0,52987.0,52952.0,...,NaN,-999,-999,25,Male,160.02,72,28.120000,112,71
152,6898d3087e9100dfd3236ee7,153,53179,53191,53197,53228,53194.0,53181.0,53207.0,53173.0,...,NaN,-999,187,23,Male,165.10,68,24.950000,132,83
181,6898e2e57e9100dfd3236f22,182,54833,54812,54799,54771,54762.0,54747.0,54765.0,54778.0,...,NaN,-999,-999,24,Male,165.10,56,20.540000,119,73
184,6898e4987e9100dfd3236f29,185,51377,51398,51376,51379,51340.0,51333.0,51317.0,51286.0,...,NaN,-999,-999,23,Male,177.80,58,18.350000,109,71
195,6898e9ff8b93e8da8a1e52ff,196,59047,59100,59002,59062,59058.0,59113.0,59148.0,59159.0,...,NaN,-999,-999,21,Male,182.88,85,25.410000,114,69
211,6899a52ce986f6946ce7d422,212,55206,55226,55206,55219,55208.0,55258.0,55314.0,55299.0,...,NaN,-999,65,25,Male,175.26,70,22.790000,114,77



Number of entries with -999 in or spo2: 17


In [ ]:
# Identify entries with spo2 less than 75% and heartRate less than 50
low_vitals_entries = df[(df['spo2'] < 80)]

# Display the identified entries
print("Entries with spo2 < 80%")
display(low_vitals_entries)

# Print the number of such entries
print(f"\nNumber of entries with spo2 < 80% {len(low_vitals_entries)}")

Entries with spo2 < 80%


,_id,serialNumber,ir[0],ir[1],ir[2],ir[3],ir[4],ir[5],ir[6],ir[7],...,red[169],spo2,heartRate,userAge,userGender,userHeight,userWeight,userBmi,systolic_BP,diastolic_BP
14,689666a4054bb9ca6f371a56,15,64927,64887,65058,65135,65135.0,65144.0,65033.0,65024.0,...,NaN,29,125,20,Male,162.00,63,24.005487,90,66
21,689666a4054bb9ca6f371a5d,22,58572,58605,58740,58810,58769.0,58645.0,58559.0,58589.0,...,NaN,51,71,22,Male,170.00,72,24.913495,151,81
25,689666a4054bb9ca6f371a61,26,53864,53871,53839,53700,53666.0,53673.0,53704.0,53683.0,...,NaN,-999,187,22,Male,167.00,70,25.099502,116,74
26,689666a4054bb9ca6f371a62,27,54786,54781,54896,54936,54974.0,54954.0,55021.0,55044.0,...,NaN,76,75,22,Male,167.00,70,25.099502,116,74
53,68971b8c6af5bb099f84c693,54,52329,52255,52322,52279,52157.0,52139.0,52285.0,52321.0,...,NaN,-999,88,21,Female,160.02,55,21.480000,112,74
56,68973056fdad67d66e5f2c03,57,59109,59134,59024,59020,59104.0,59088.0,59110.0,59083.0,...,NaN,58,107,22,Male,167.64,65,23.130000,100,66
59,6897385157f90f6e80917148,60,56938,56954,56909,56921,56961.0,56968.0,56911.0,56969.0,...,NaN,-999,115,25,Male,170.18,76,26.240000,120,80
66,68973cd9f077907221199206,67,56054,56063,56044,56060,56056.0,56047.0,56073.0,56075.0,...,NaN,72,214,22,Male,172.72,74,24.810000,141,81
91,68979520341377cae496fbf1,92,51691,51713,51515,51569,51567.0,51465.0,51549.0,51529.0,...,NaN,75,93,21,Male,172.72,64,21.450000,114,84
118,6898b131663f697566990966,119,59568,59534,59551,59499,59504.0,59548.0,59530.0,59509.0,...,NaN,75,187,21,Male,175.26,85,27.670000,139,103



Number of entries with spo2 < 80% 37


In [ ]:
# Identify entries with spo2 less than 75% and heartRate less than 50
low_vitals_entries = df[(df['heartRate'] < 50)]

# Display the identified entries
print("Entries with and heartRate < 50:")
display(low_vitals_entries)

# Print the number of such entries
print(f"\nNumber of entries with heartRate < 50: {len(low_vitals_entries)}")

Entries with and heartRate < 50:


,_id,serialNumber,ir[0],ir[1],ir[2],ir[3],ir[4],ir[5],ir[6],ir[7],...,red[169],spo2,heartRate,userAge,userGender,userHeight,userWeight,userBmi,systolic_BP,diastolic_BP
141,6898ce537e9100dfd3236ed0,142,52901,52895,52882,52882,52859.0,52845.0,52849.0,52845.0,...,NaN,-999,-999,22,Male,180.34,69,21.22,108,75
149,6898d1b77e9100dfd3236ee0,150,52939,52974,52996,52782,52887.0,52967.0,52987.0,52952.0,...,NaN,-999,-999,25,Male,160.02,72,28.12,112,71
181,6898e2e57e9100dfd3236f22,182,54833,54812,54799,54771,54762.0,54747.0,54765.0,54778.0,...,NaN,-999,-999,24,Male,165.10,56,20.54,119,73
184,6898e4987e9100dfd3236f29,185,51377,51398,51376,51379,51340.0,51333.0,51317.0,51286.0,...,NaN,-999,-999,23,Male,177.80,58,18.35,109,71
195,6898e9ff8b93e8da8a1e52ff,196,59047,59100,59002,59062,59058.0,59113.0,59148.0,59159.0,...,NaN,-999,-999,21,Male,182.88,85,25.41,114,69
218,6899a921e986f6946ce7d432,219,58574,58394,58372,58441,58415.0,58306.0,58316.0,58348.0,...,NaN,99,48,25,Male,165.10,72,26.41,118,84
220,6899aa06e986f6946ce7d436,221,54206,54328,54274,54246,54403.0,54357.0,54302.0,54343.0,...,NaN,99,48,23,Female,172.72,65,21.79,107,76
221,6899aa16e986f6946ce7d437,222,54206,54328,54274,54246,54403.0,54357.0,54302.0,54343.0,...,NaN,99,48,23,Female,172.72,65,21.79,107,76
236,6899b3b9e986f6946ce7d45a,237,54231,54219,54217,54213,54192.0,54204.0,54195.0,54202.0,...,NaN,-999,-999,21,Female,165.10,70,25.68,115,85
241,6899b6bce986f6946ce7d466,242,59814,59802,59822,59806,59809.0,59843.0,59824.0,59841.0,...,NaN,-999,-999,21,Female,165.10,70,25.68,115,85



Number of entries with heartRate < 50: 17


In [ ]:
# Identify entries with spo2 less than 75% and heartRate less than 50
low_vitals_entries = df[(df['spo2'] < 80) | (df['heartRate'] < 50) ]

# Display the identified entries
print("Entries with spo2 < 80% and or heartRate < 50")
display(low_vitals_entries)

# Print the number of such entries
print(f"\nNumber of entries with spo2 < 80% and or heartRate < 50: {len(low_vitals_entries)}")

Entries with spo2 < 80% and or heartRate < 50


,_id,serialNumber,ir[0],ir[1],ir[2],ir[3],ir[4],ir[5],ir[6],ir[7],...,red[169],spo2,heartRate,userAge,userGender,userHeight,userWeight,userBmi,systolic_BP,diastolic_BP



Number of entries with spo2 < 80% and or heartRate < 50: 0


In [ ]:
# Impute `spo2` values:
# Replace -999 and values < 80 with NaN
df['spo2'] = df['spo2'].replace(-999, np.nan)
df['spo2'] = df['spo2'].where(df['spo2'] >= 80, np.nan)  # Replace values below 80 with NaN
spo2_median = df['spo2'].median()  # Get the median
df['spo2'].fillna(spo2_median, inplace=True)  # Impute missing values with the median

# Impute `heartRate` values:
# Replace -999 and values < 45 with NaN
df['heartRate'] = df['heartRate'].replace(-999, np.nan)
df['heartRate'] = df['heartRate'].where(df['heartRate'] >= 50, np.nan)  # Replace values below 45 with NaN
heartRate_mean = df['heartRate'].mean()  # Get the mean
df['heartRate'].fillna(heartRate_mean, inplace=True)  # Impute missing values with the mean

# Check the data after imputation
print(df[['spo2', 'heartRate']].describe())


             spo2   heartRate
count  328.000000  328.000000
mean    97.106707  140.926045
std      4.305343   44.097241
min     80.000000   50.000000
25%     96.750000  107.000000
50%     99.000000  140.926045
75%    100.000000  166.000000
max    100.000000  300.000000


/tmp/ipython-input-2736165181.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['spo2'].fillna(spo2_median, inplace=True)  # Impute missing values with the median
/tmp/ipython-input-2736165181.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col

In [ ]:
# Identify entries with spo2 less than 75% and heartRate less than 50
high_vitals_entries = df[ (df['heartRate'] >= 200) ]

# Display the identified entries
print("Entries with heartRate => 50")
display(high_vitals_entries)

# Print the number of such entries
print(f"\nNumber of entries heartRate => 50: {len(high_vitals_entries)}")

Entries with heartRate => 50


,_id,serialNumber,ir[0],ir[1],ir[2],ir[3],ir[4],ir[5],ir[6],ir[7],...,red[169],spo2,heartRate,userAge,userGender,userHeight,userWeight,userBmi,systolic_BP,diastolic_BP
12,689666a4054bb9ca6f371a54,13,66466,66424,66429,66419,66420.0,66444.0,66415.0,66472.0,...,NaN,99.0,250.0,21,Female,165.00,49,17.998163,110,78
13,689666a4054bb9ca6f371a55,14,67309,67167,67002,67128,67186.0,67090.0,67016.0,66997.0,...,NaN,95.0,214.0,24,Male,172.00,60,20.281233,105,80
15,689666a4054bb9ca6f371a57,16,63208,63242,63070,63053,63202.0,63283.0,63379.0,63264.0,...,NaN,88.0,250.0,20,Male,172.00,54,18.253110,123,82
66,68973cd9f077907221199206,67,56054,56063,56044,56060,56056.0,56047.0,56073.0,56075.0,...,NaN,99.0,214.0,22,Male,172.72,74,24.810000,141,81
74,689747cf48323f172b3fd599,75,58665,58626,58626,58649,58602.0,58593.0,58581.0,58565.0,...,NaN,82.0,214.0,27,Male,172.72,89,29.830000,116,73
75,6897486548323f172b3fd59b,76,62058,62147,62312,62497,62612.0,62574.0,62459.0,62594.0,...,NaN,95.0,250.0,19,Male,175.26,64,20.840000,108,67
109,6898642c439b265a0f852ac0,110,57606,57589,57349,57344,57447.0,57213.0,57161.0,57188.0,...,NaN,99.0,214.0,25,Male,172.72,80,26.820000,118,86
110,68986491439b265a0f852ac2,111,57124,57128,57090,57006,56928.0,56899.0,56882.0,56786.0,...,NaN,81.0,214.0,26,Male,177.80,86,27.200000,132,84
111,68986536439b265a0f852ac5,112,55072,54944,55222,55174,54911.0,55086.0,55700.0,55598.0,...,NaN,96.0,214.0,24,Male,167.64,75,26.690000,107,63
112,68986541439b265a0f852ac6,113,55072,54944,55222,55174,54911.0,55086.0,55700.0,55598.0,...,NaN,96.0,214.0,24,Male,167.64,75,26.690000,107,63



Number of entries heartRate => 50: 30


In [ ]:
# Replace heartRate values above a certain threshold (e.g., 200 bpm) with NaN
df['heartRate'] = df['heartRate'].replace(-999, np.nan)  # Replace -999 with NaN if not already done
df['heartRate'] = df['heartRate'].where(df['heartRate'] < 200, np.nan)  # Replace extreme heart rate with NaN

# Now impute these NaN values with the mean or median
heartRate_median = df['heartRate'].median()  # Get the median of heart rate
df['heartRate'].fillna(heartRate_median, inplace=True)  # Impute with the median

# Check the data after imputation
print(df['heartRate'].describe())


count    328.000000
mean     132.389460
std       33.185454
min       50.000000
25%      107.000000
50%      136.000000
75%      150.000000
max      187.000000
Name: heartRate, dtype: float64


/tmp/ipython-input-1209852446.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['heartRate'].fillna(heartRate_median, inplace=True)  # Impute with the median


In [ ]:
# Calculate the median for both 'heartRate' and 'spo2' columns
heart_rate_median = df['heartRate'].median()
spo2_median = df['spo2'].median()

print(f"Median heartRate: {heart_rate_median}")
print(f"Median spo2: {spo2_median}")


Median heartRate: 136.0
Median spo2: 99.0


In [ ]:
# Save the DataFrame to a CSV file
output_path = "df_with_nulls.csv" # You can change the filename
df.to_csv(output_path, index=False)

print(f"DataFrame saved to '{output_path}'")

DataFrame saved to 'df_with_nulls.csv'


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Initialize the LabelEncoder
le = LabelEncoder()

# Label encode the gender column
df['userGender'] = le.fit_transform(df['userGender'])

# Check the result
print(df['userGender'].head())

# Optionally, check the mapping of labels to values
print(le.classes_)  # This will print the unique categories mapped to 0, 1, 2, etc.


0    1
1    0
2    0
3    1
4    1
Name: userGender, dtype: int64
['Female' 'Male']


In [ ]:
display(df.head(15))

,_id,serialNumber,ir[0],ir[1],ir[2],ir[3],ir[4],ir[5],ir[6],ir[7],...,red[169],spo2,heartRate,userAge,userGender,userHeight,userWeight,userBmi,systolic_BP,diastolic_BP
0,689666a4054bb9ca6f371a48,1,53465,53297,53166,53039,53234.0,53247.0,53156.0,53171.0,...,NaN,100.0,107.0,44,1,180.0,98,30.246914,122,83
1,689666a4054bb9ca6f371a49,2,54755,54760,54767,54775,54782.0,54785.0,54804.0,54817.0,...,30759.0,100.0,125.0,30,0,154.0,43,18.131219,102,68
2,689666a4054bb9ca6f371a4a,3,61920,61724,61878,61801,61611.0,61651.0,61801.0,61822.0,...,NaN,100.0,93.0,61,0,144.0,62,29.899691,141,93
3,689666a4054bb9ca6f371a4b,4,59870,59829,59868,59922,59982.0,59954.0,59904.0,59956.0,...,NaN,98.0,71.0,45,1,172.0,103,34.816117,139,96
4,689666a4054bb9ca6f371a4c,5,61487,61599,61710,61695,61531.0,61550.0,61506.0,61582.0,...,45313.0,98.0,166.0,23,1,172.0,88,29.745809,112,75
5,689666a4054bb9ca6f371a4d,6,61878,62116,62117,62150,62085.0,62098.0,62093.0,62133.0,...,NaN,100.0,107.0,21,1,165.0,68,24.977043,104,61
6,689666a4054bb9ca6f371a4e,7,55798,55766,55710,55678,55677.0,55689.0,55689.0,55698.0,...,NaN,100.0,136.0,22,1,167.0,86,30.836531,108,80
7,689666a4054bb9ca6f371a4f,8,65781,65850,65149,65549,65698.0,65876.0,65877.0,65411.0,...,NaN,100.0,150.0,23,1,177.0,107,34.153660,111,72
8,689666a4054bb9ca6f371a50,9,62554,2783,1136,1148,24925.0,58433.0,59563.0,60109.0,...,NaN,99.0,68.0,21,1,182.0,100,30.189591,120,79
10,689666a4054bb9ca6f371a52,11,59696,59669,59694,59683,59732.0,59569.0,59619.0,59619.0,...,NaN,100.0,115.0,21,1,180.0,78,24.074074,116,68


In [ ]:
df = df.drop(columns=['_id', 'serialNumber'])

# To verify
print(df.head())


   ir[0]  ir[1]  ir[2]  ir[3]    ir[4]    ir[5]    ir[6]    ir[7]    ir[8]  \
0  53465  53297  53166  53039  53234.0  53247.0  53156.0  53171.0  53147.0   
1  54755  54760  54767  54775  54782.0  54785.0  54804.0  54817.0  54858.0   
2  61920  61724  61878  61801  61611.0  61651.0  61801.0  61822.0  61699.0   
3  59870  59829  59868  59922  59982.0  59954.0  59904.0  59956.0  59963.0   
4  61487  61599  61710  61695  61531.0  61550.0  61506.0  61582.0  61451.0   

     ir[9]  ...  red[169]   spo2  heartRate  userAge  userGender  userHeight  \
0  53255.0  ...       NaN  100.0      107.0       44           1       180.0   
1  54864.0  ...   30759.0  100.0      125.0       30           0       154.0   
2  61829.0  ...       NaN  100.0       93.0       61           0       144.0   
3  59986.0  ...       NaN   98.0       71.0       45           1       172.0   
4  61508.0  ...   45313.0   98.0      166.0       23           1       172.0   

   userWeight    userBmi  systolic_BP  diastolic_B

In [ ]:
print(f"New DataFrame shape: {df.shape}")

New DataFrame shape: (328, 349)


In [ ]:
# Check for duplicate rows in df_limited
duplicate_rows = df[df.duplicated()]

# Display duplicate rows
display(duplicate_rows)

# Print the number of duplicate rows
print(f"\nNumber of duplicate entries in df_limited: {len(duplicate_rows)}")

,ir[0],ir[1],ir[2],ir[3],ir[4],ir[5],ir[6],ir[7],ir[8],ir[9],...,red[169],spo2,heartRate,userAge,userGender,userHeight,userWeight,userBmi,systolic_BP,diastolic_BP
112,55072,54944,55222,55174,54911.0,55086.0,55700.0,55598.0,55278.0,55090.0,...,NaN,96.0,136.000000,24,1,167.64,75,26.69,107,63
146,56031,56093,56053,55953,55940.0,55911.0,55902.0,55896.0,55809.0,55699.0,...,NaN,89.0,150.000000,25,1,180.34,82,25.21,108,70
154,54717,54334,54519,54577,54714.0,54793.0,54670.0,54483.0,54571.0,54654.0,...,NaN,100.0,107.000000,23,1,180.34,54,16.60,115,80
189,53802,53850,53569,53625,53686.0,53786.0,53580.0,53408.0,53432.0,53458.0,...,NaN,92.0,150.000000,23,1,175.26,48,15.63,113,61
221,54206,54328,54274,54246,54403.0,54357.0,54302.0,54343.0,54277.0,54230.0,...,NaN,99.0,140.926045,23,0,172.72,65,21.79,107,76
235,51965,52070,51590,51714,51848.0,51991.0,51552.0,51699.0,51841.0,52036.0,...,NaN,100.0,150.000000,22,0,154.94,55,22.91,107,74
245,47397,47427,47574,47806,48133.0,48225.0,48127.0,48096.0,47973.0,47932.0,...,NaN,100.0,150.000000,22,0,154.94,56,23.33,110,78
249,49065,49049,48969,49039,49025.0,49084.0,49121.0,49045.0,49083.0,49095.0,...,NaN,99.0,166.000000,48,0,152.40,58,24.97,119,70
252,52322,52396,52220,52171,52252.0,52279.0,52215.0,52104.0,52155.0,52127.0,...,NaN,99.0,150.000000,47,0,160.02,69,26.95,127,86
259,57870,57955,58018,58083,58139.0,58269.0,58984.0,59231.0,58514.0,57675.0,...,NaN,100.0,166.000000,36,0,157.48,50,20.16,133,71



Number of duplicate entries in df_limited: 11


In [ ]:
# Remove duplicate rows
df_cleaned = df.drop_duplicates()

# Check the number of rows after dropping duplicates
print(f"Rows after removing duplicates: {df_cleaned.shape[0]}")


Rows after removing duplicates: 317


In [ ]:
# Check for duplicate rows in df_limited
duplicate_rows = df_cleaned[df_cleaned.duplicated()]

# Display duplicate rows
display(duplicate_rows)

# Print the number of duplicate rows
print(f"\nNumber of duplicate entries in df_limited: {len(duplicate_rows)}")

,ir[0],ir[1],ir[2],ir[3],ir[4],ir[5],ir[6],ir[7],ir[8],ir[9],...,red[169],spo2,heartRate,userAge,userGender,userHeight,userWeight,userBmi,systolic_BP,diastolic_BP



Number of duplicate entries in df_limited: 0


In [ ]:

# --- STEP 3: Identify IR and RED columns ---
ir_cols = [c for c in df_cleaned.columns if c.startswith("ir[")]
red_cols = [c for c in df_cleaned.columns if c.startswith("red[")]

In [ ]:
# --- STEP 3.1: Check the number of non-null values in IR and RED arrays ---

# Get the number of non-null values for each row in IR columns
ir_counts = df_cleaned[ir_cols].notna().sum(axis=1)

# Get the number of non-null values for each row in RED columns
red_counts = df_cleaned[red_cols].notna().sum(axis=1)

# Find the minimum and maximum counts for IR
min_ir_count = ir_counts.min()
max_ir_count = ir_counts.max()

# Find the minimum and maximum counts for RED
min_red_count = red_counts.min()
max_red_count = red_counts.max()

print(f"Minimum number of non-null values in IR arrays: {min_ir_count}")
print(f"Maximum number of non-null values in IR arrays: {max_ir_count}")
print(f"Minimum number of non-null values in RED arrays: {min_red_count}")
print(f"Maximum number of non-null values in RED arrays: {max_red_count}")

Minimum number of non-null values in IR arrays: 113
Maximum number of non-null values in IR arrays: 170
Minimum number of non-null values in RED arrays: 113
Maximum number of non-null values in RED arrays: 170


In [ ]:
# --- STEP 4: Function to resample waveform to fixed length ---
def resample_to_fixed_length(row, cols, target_len=200):
    arr = row[cols].to_numpy(dtype=float)
    # Remove NaNs before resampling (shorten series if needed)
    arr = arr[~np.isnan(arr)]
    if len(arr) == 0:
        return np.full(target_len, np.nan)  # empty signal
    return resample(arr, target_len)


In [ ]:
# --- STEP 5: Apply resampling to IR and RED ---
target_length = 200
ir_resampled = np.vstack(df_cleaned.apply(lambda row: resample_to_fixed_length(row, ir_cols, target_length), axis=1))
red_resampled = np.vstack(df_cleaned.apply(lambda row: resample_to_fixed_length(row, red_cols, target_length), axis=1))

In [ ]:
# --- STEP 6: Save resampled signals into DataFrame ---
ir_df = pd.DataFrame(ir_resampled, columns=[f"ir_{i}" for i in range(target_length)])
red_df = pd.DataFrame(red_resampled, columns=[f"red_{i}" for i in range(target_length)])

In [ ]:
# --- STEP 7: Merge back with meta + labels ---
keep_cols = ['spo2', 'heartRate', 'userAge', 'userGender', 'userHeight', 'userWeight', 'userBmi', 'systolic_BP', 'diastolic_BP']
df_clean = pd.concat([df_cleaned[keep_cols].reset_index(drop=True), ir_df, red_df], axis=1)

In [ ]:
# --- STEP 8: Save cleaned dataset ---
output_path = "/content/bp_dataset_resampledV2.csv"
df_clean.to_csv(output_path, index=False)
print(f"✅ Cleaned & resampled dataset saved to: {output_path}")
print(f"Shape: {df_clean.shape}")


✅ Cleaned & resampled dataset saved to: /content/bp_dataset_resampledV2.csv
Shape: (317, 409)


In [ ]:
display(df_clean.head())

,spo2,heartRate,userAge,userGender,userHeight,userWeight,userBmi,systolic_BP,diastolic_BP,ir_0,...,red_190,red_191,red_192,red_193,red_194,red_195,red_196,red_197,red_198,red_199
0,100.0,107.0,44,1,180.0,98,30.246914,122,83,53465.0,...,27273.118598,27284.864828,27238.849462,27305.232194,27289.679810,27243.852603,27283.764741,27269.051700,27225.870816,27214.390034
1,100.0,125.0,30,0,154.0,43,18.131219,102,68,54755.0,...,30849.143333,30769.175435,30814.321042,30766.920469,30744.651786,30805.381260,30786.732599,30850.532347,30755.787616,30799.954928
2,100.0,93.0,61,0,144.0,62,29.899691,141,93,61920.0,...,44723.357747,44687.385179,44752.517717,44723.396119,44683.442669,44709.866599,44697.651598,44764.138740,44684.514943,44734.477694
3,98.0,71.0,45,1,172.0,103,34.816117,139,96,59870.0,...,38796.905896,38803.597493,38781.293654,38783.916183,38796.831129,38806.214866,38822.492875,38788.759159,38788.628134,38766.401531
4,98.0,166.0,23,1,172.0,88,29.745809,112,75,61487.0,...,45363.027161,45331.427991,45325.212267,45321.077108,45340.601187,45361.981611,45325.363942,45355.593678,45231.186328,45388.795779


In [ ]:
# --- Step 1: Extract the relevant columns for IR and RED signals ---
ir_cols = [c for c in df_clean.columns if c.startswith("ir_")]
red_cols = [c for c in df_clean.columns if c.startswith("red_")]


In [ ]:
from scipy.stats import skew, kurtosis
from scipy.signal import find_peaks
from scipy.fft import fft

In [ ]:
# --- Function to extract time-domain and frequency-domain features from a signal ---
def extract_features(signal):
    # Ensure the signal is a float numpy array
    signal = np.asarray(signal, dtype=float)

    # Handle cases with all NaN values after conversion
    if np.all(np.isnan(signal)):
        # Return a list of NaNs for all features if the signal is all NaNs
        return [np.nan] * 14 # 14 is the number of features being extracted

    # Time-domain features
    mean_signal = np.nanmean(signal) # Use nanmean to ignore NaNs
    median_signal = np.nanmedian(signal) # Use nanmedian to ignore NaNs
    std_signal = np.nanstd(signal) # Use nanstd to ignore NaNs
    min_signal = np.nanmin(signal) if not np.all(np.isnan(signal)) else np.nan # Handle min/max on all NaNs
    max_signal = np.nanmax(signal) if not np.all(np.isnan(signal)) else np.nan # Handle min/max on all NaNs
    range_signal = max_signal - min_signal if not np.all(np.isnan(signal)) else np.nan # Handle range on all NaNs

    # SciPy stats functions handle NaNs with nan_policy='omit' or 'propagate'
    # Let's ensure we handle potential NaNs explicitly or rely on default nan_policy
    skewness_signal = skew(signal, nan_policy='omit') if len(signal[~np.isnan(signal)]) > 1 else np.nan # Need at least 2 non-NaN for skew
    kurtosis_signal = kurtosis(signal, nan_policy='omit') if len(signal[~np.isnan(signal)]) > 3 else np.nan # Need at least 4 non-NaN for kurtosis


    # Peak features (find peaks in the signal) - find_peaks requires non-NaN input
    non_nan_signal = signal[~np.isnan(signal)]
    peaks, _ = find_peaks(non_nan_signal)
    num_peaks = len(peaks)
    mean_peak_amplitude = np.nanmean(non_nan_signal[peaks]) if num_peaks > 0 else np.nan # Use nanmean

    # Frequency-domain features using FFT - fft requires non-NaN input
    if len(non_nan_signal) == 0:
         return [np.nan] * 14 # Return NaNs if no non-NaN data for FFT

    fft_values = fft(non_nan_signal)
    fft_freq = np.fft.fftfreq(len(non_nan_signal))
    fft_magnitude = np.abs(fft_values)

    # Extract dominant frequency (proxy for heart rate)
    # Need to handle case where all magnitudes are zero (e.g., constant signal)
    if np.all(fft_magnitude[1:] == 0):
         dominant_freq = np.nan
    else:
        dominant_freq = np.abs(fft_freq[np.argmax(fft_magnitude[1:]) + 1])  # Skip zero frequency (DC component)


    # Total spectral energy
    total_energy = np.sum(fft_magnitude**2)

    # Low/High Frequency Band power - ensure frequency bounds are within fft_freq range
    low_freq_indices = np.where((fft_freq >= 0.5) & (fft_freq <= 3))[0]
    high_freq_indices = np.where((fft_freq >= 3) & (fft_freq <= 6))[0]

    low_freq_band = np.sum(fft_magnitude[low_freq_indices])
    high_freq_band = np.sum(fft_magnitude[high_freq_indices])


    return [
        mean_signal, median_signal, std_signal, min_signal, max_signal, range_signal,
        skewness_signal, kurtosis_signal, num_peaks, mean_peak_amplitude,
        dominant_freq, total_energy, low_freq_band, high_freq_band
    ]

In [ ]:
# --- Extract features for both IR and Red signals ---
def extract_features_from_df(df, ir_columns, red_columns):
    features = []
    for i, row in df_clean.iterrows():
        # Extract IR and Red signal features
        ir_signal = row[ir_columns].values
        red_signal = row[red_columns].values

        ir_features = extract_features(ir_signal)
        red_features = extract_features(red_signal)

        # Combine IR and Red features into one set
        combined_features = ir_features + red_features

        # Append demographics
        demographics = [
            row['userAge'], row['userGender'], row['userHeight'], row['userWeight'],
            row['userBmi'], row['heartRate'], row['spo2']  # Assuming these columns are present
        ]

        features.append(combined_features + demographics)

    return np.array(features)


In [ ]:
# --- Prepare feature matrix and labels ---
X = extract_features_from_df(df_clean, ir_cols, red_cols)


In [ ]:
# --- Labels (systolic_BP, diastolic_BP) ---
y = df_clean[['systolic_BP', 'diastolic_BP']].values

In [ ]:
# --- Create a DataFrame for the features ---
columns = [
    'ir_mean', 'ir_median', 'ir_std', 'ir_min', 'ir_max', 'ir_range', 'ir_skewness', 'ir_kurtosis',
    'ir_num_peaks', 'ir_mean_peak_amplitude', 'ir_dominant_freq', 'ir_total_energy', 'ir_low_freq_band', 'ir_high_freq_band',
    'red_mean', 'red_median', 'red_std', 'red_min', 'red_max', 'red_range', 'red_skewness', 'red_kurtosis',
    'red_num_peaks', 'red_mean_peak_amplitude', 'red_dominant_freq', 'red_total_energy', 'red_low_freq_band', 'red_high_freq_band',
    'userAge', 'userGender', 'userHeight', 'userWeight', 'userBmi', 'heartRate', 'spo2'
]

In [ ]:
X_df = pd.DataFrame(X, columns=columns)

In [ ]:
display(X_df.head())

,ir_mean,ir_median,ir_std,ir_min,ir_max,ir_range,ir_skewness,ir_kurtosis,ir_num_peaks,ir_mean_peak_amplitude,...,red_total_energy,red_low_freq_band,red_high_freq_band,userAge,userGender,userHeight,userWeight,userBmi,heartRate,spo2
0,52577.426036,52672.286147,494.408818,51214.295208,53481.287901,2266.992693,-0.679211,-0.234299,56.0,52712.770458,...,2.982998e+13,0.0,0.0,44.0,1.0,180.0,98.0,30.246914,107.0,100.0
1,54981.852941,55090.747907,331.865852,52820.784885,55337.866131,2517.081246,-3.197905,13.477465,55.0,55092.154130,...,3.837467e+13,0.0,0.0,30.0,0.0,154.0,43.0,18.131219,125.0,100.0
2,61247.928994,61100.751013,344.706523,60731.276101,61941.177431,1209.901330,0.327449,-1.373854,54.0,61369.849652,...,8.062040e+13,0.0,0.0,61.0,0.0,144.0,62.0,29.899691,93.0,100.0
3,60305.934911,60340.486227,213.482900,59800.345442,60683.493785,883.148344,-0.406957,-0.882666,41.0,60349.452409,...,6.006732e+13,0.0,0.0,45.0,1.0,172.0,103.0,34.816117,71.0,98.0
4,60658.988235,60527.884581,324.515476,60259.382240,61786.899002,1527.516763,1.459666,1.720295,51.0,60724.343618,...,8.233843e+13,0.0,0.0,23.0,1.0,172.0,88.0,29.745809,166.0,98.0


In [ ]:
print(f"Shape of features: {X_df.shape}")
print(f"Shape of labels: {y.shape}")

Shape of features: (317, 35)
Shape of labels: (317, 2)


In [ ]:
# Extract features for each row in IR and RED signal columns
def extract_features_from_df(df, ir_columns, red_columns):
    features = []
    for i, row in df_clean.iterrows():
        # Extract IR signal features
        ir_signal = row[ir_columns].values
        red_signal = row[red_columns].values

        # Extract features from both IR and RED signals
        ir_features = extract_features(ir_signal)
        red_features = extract_features(red_signal)

        # Combine IR and RED features
        combined_features = ir_features + red_features

        # Append the combined features to the list
        features.append(combined_features)

    return np.array(features)

# Apply the function to extract features from the dataset
ir_features_df = extract_features_from_df(df_clean, ir_cols, red_cols)

# Convert the extracted features into a DataFrame for easier manipulation
columns = [
    'ir_mean', 'ir_median', 'ir_std', 'ir_min', 'ir_max', 'ir_range', 'ir_skewness', 'ir_kurtosis',
    'ir_num_peaks', 'ir_mean_peak_amplitude', 'ir_dominant_freq', 'ir_total_energy', 'ir_low_freq_band', 'ir_high_freq_band',
    'red_mean', 'red_median', 'red_std', 'red_min', 'red_max', 'red_range', 'red_skewness', 'red_kurtosis',
    'red_num_peaks', 'red_mean_peak_amplitude', 'red_dominant_freq', 'red_total_energy', 'red_low_freq_band', 'red_high_freq_band'
]

# Create a DataFrame with the extracted features
features_df = pd.DataFrame(ir_features_df, columns=columns)

# Check the first few rows of the extracted features DataFrame
print(features_df.head())


        ir_mean     ir_median      ir_std        ir_min        ir_max  \
0  52577.426036  52672.286147  494.408818  51214.295208  53481.287901   
1  54981.852941  55090.747907  331.865852  52820.784885  55337.866131   
2  61247.928994  61100.751013  344.706523  60731.276101  61941.177431   
3  60305.934911  60340.486227  213.482900  59800.345442  60683.493785   
4  60658.988235  60527.884581  324.515476  60259.382240  61786.899002   

      ir_range  ir_skewness  ir_kurtosis  ir_num_peaks  \
0  2266.992693    -0.679211    -0.234299          56.0   
1  2517.081246    -3.197905    13.477465          55.0   
2  1209.901330     0.327449    -1.373854          54.0   
3   883.148344    -0.406957    -0.882666          41.0   
4  1527.516763     1.459666     1.720295          51.0   

   ir_mean_peak_amplitude  ...       red_max   red_range  red_skewness  \
0            52712.770458  ...  27510.401942  424.010454      0.106929   
1            55092.154130  ...  31132.194630  852.277107     -1.

In [ ]:
print(f"Shape of features: {features_df.shape}")
print(f"Shape of labels: {y.shape}")

Shape of features: (317, 28)
Shape of labels: (317, 2)


In [ ]:
features_df = features_df.drop(columns=['ir_low_freq_band', 'ir_high_freq_band', 'red_low_freq_band', 'red_high_freq_band'])


KeyError: "['ir_low_freq_band', 'ir_high_freq_band', 'red_low_freq_band', 'red_high_freq_band'] not found in axis"

In [ ]:

# To verify
print(features_df.head())

        ir_mean     ir_median      ir_std        ir_min        ir_max  \
0  52577.426036  52672.286147  494.408818  51214.295208  53481.287901   
1  54981.852941  55090.747907  331.865852  52820.784885  55337.866131   
2  61247.928994  61100.751013  344.706523  60731.276101  61941.177431   
3  60305.934911  60340.486227  213.482900  59800.345442  60683.493785   
4  60658.988235  60527.884581  324.515476  60259.382240  61786.899002   

      ir_range  ir_skewness  ir_kurtosis  ir_num_peaks  \
0  2266.992693    -0.679211    -0.234299          56.0   
1  2517.081246    -3.197905    13.477465          55.0   
2  1209.901330     0.327449    -1.373854          54.0   
3   883.148344    -0.406957    -0.882666          41.0   
4  1527.516763     1.459666     1.720295          51.0   

   ir_mean_peak_amplitude  ...     red_std       red_min       red_max  \
0            52712.770458  ...   78.286441  27086.391488  27510.401942   
1            55092.154130  ...  151.642262  30279.917522  31132.

In [ ]:
# Combine the features with demographics and vitals
all_features_df = pd.concat([features_df, df_clean[demographics_vitals_cols]], axis=1)
print("Shape of all_features_df:", all_features_df.shape)
display(all_features_df.head())


Shape of all_features_df: (317, 38)


,ir_mean,ir_median,ir_std,ir_min,ir_max,ir_range,ir_skewness,ir_kurtosis,ir_num_peaks,ir_mean_peak_amplitude,...,userBmi,heartRate,spo2,userAge,userGender,userHeight,userWeight,userBmi,spo2,heartRate
0,52577.426036,52672.286147,494.408818,51214.295208,53481.287901,2266.992693,-0.679211,-0.234299,56.0,52712.770458,...,30.246914,107.0,100.0,44,1,180.0,98,30.246914,100.0,107.0
1,54981.852941,55090.747907,331.865852,52820.784885,55337.866131,2517.081246,-3.197905,13.477465,55.0,55092.154130,...,18.131219,125.0,100.0,30,0,154.0,43,18.131219,100.0,125.0
2,61247.928994,61100.751013,344.706523,60731.276101,61941.177431,1209.901330,0.327449,-1.373854,54.0,61369.849652,...,29.899691,93.0,100.0,61,0,144.0,62,29.899691,100.0,93.0
3,60305.934911,60340.486227,213.482900,59800.345442,60683.493785,883.148344,-0.406957,-0.882666,41.0,60349.452409,...,34.816117,71.0,98.0,45,1,172.0,103,34.816117,98.0,71.0
4,60658.988235,60527.884581,324.515476,60259.382240,61786.899002,1527.516763,1.459666,1.720295,51.0,60724.343618,...,29.745809,166.0,98.0,23,1,172.0,88,29.745809,98.0,166.0


In [ ]:
ir_cols = [c for c in features_df.columns if c.startswith("ir_")]
red_cols = [c for c in features_df.columns if c.startswith("red_")]

# Processing only IR

In [ ]:
# Combine the IR features with demographics and vitals
ir_features_with_demographics_vitals_df = pd.concat([features_df[ir_cols], df_clean[demographics_vitals_cols]], axis=1)
print("Shape of ir_features_with_demographics_vitals_df:", ir_features_with_demographics_vitals_df.shape)
display(ir_features_with_demographics_vitals_df)

Shape of ir_features_with_demographics_vitals_df: (317, 19)


,ir_mean,ir_median,ir_std,ir_min,ir_max,ir_range,ir_skewness,ir_kurtosis,ir_num_peaks,ir_mean_peak_amplitude,ir_dominant_freq,ir_total_energy,userAge,userGender,userHeight,userWeight,userBmi,spo2,heartRate
0,52577.426036,52672.286147,494.408818,51214.295208,53481.287901,2266.992693,-0.679211,-0.234299,56.0,52712.770458,0.005,1.105852e+14,44,1,180.00,98,30.246914,100.0,107.000000
1,54981.852941,55090.747907,331.865852,52820.784885,55337.866131,2517.081246,-3.197905,13.477465,55.0,55092.154130,0.005,1.209246e+14,30,0,154.00,43,18.131219,100.0,125.000000
2,61247.928994,61100.751013,344.706523,60731.276101,61941.177431,1209.901330,0.327449,-1.373854,54.0,61369.849652,0.005,1.500571e+14,61,0,144.00,62,29.899691,100.0,93.000000
3,60305.934911,60340.486227,213.482900,59800.345442,60683.493785,883.148344,-0.406957,-0.882666,41.0,60349.452409,0.005,1.454741e+14,45,1,172.00,103,34.816117,98.0,71.000000
4,60658.988235,60527.884581,324.515476,60259.382240,61786.899002,1527.516763,1.459666,1.720295,51.0,60724.343618,0.005,1.471847e+14,23,1,172.00,88,29.745809,98.0,166.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
312,59224.005917,59448.390741,788.632968,56790.056344,60195.235560,3405.179217,-1.394538,1.026251,44.0,59323.088459,0.005,1.403242e+14,23,1,170.18,57,19.680000,96.0,65.000000
313,51116.502959,51114.711564,89.709731,50915.891367,51307.404803,391.513435,0.050864,-0.697014,49.0,51151.176904,0.010,1.045162e+14,27,1,170.18,70,24.170000,94.0,136.000000
314,52143.810651,52194.061032,288.178486,51516.752536,52562.478997,1045.726461,-0.578628,-0.985078,53.0,52183.504303,0.005,1.087624e+14,26,1,177.80,85,26.890000,94.0,136.000000
315,55118.153846,55362.143367,692.900985,53449.629824,55880.759455,2431.129631,-0.669441,-1.004606,35.0,55309.535752,0.005,1.215396e+14,25,1,175.26,58,18.880000,97.0,136.000000
